In [9]:
#!/usr/bin/env python
# -*- coding: utf-8 -*-

"""
Phase 1: Core Functions and Data Loading for Yearly Drought Maps with NetCDF Output
This phase includes the essential functions for loading data and basic processing.
"""

import os
import sys
import time
import logging
import numpy as np
import xarray as xr
import pandas as pd
import calendar
from datetime import datetime
import matplotlib.pyplot as plt
import matplotlib.colors as mcolors
import matplotlib.cm as cm
import cartopy.crs as ccrs
import cartopy.feature as cfeature
from cartopy.mpl.gridliner import LONGITUDE_FORMATTER, LATITUDE_FORMATTER
from tqdm import tqdm

# Configure logging
logging.basicConfig(
    level=logging.INFO,
    format="%(asctime)s [%(levelname)s] %(message)s",
    datefmt="%Y-%m-%d %H:%M:%S",
    handlers=[
        logging.StreamHandler(sys.stdout),
        logging.FileHandler("yearly_drought_maps.log")
    ]
)
logger = logging.getLogger(__name__)

def load_cdi_data(file_path, end_year=2018):
    """
    Load the CDI file and extract data up to the specified end year.
    
    Args:
        file_path (str): Path to the CDI file
        end_year (int): End year for the data (inclusive)
        
    Returns:
        xarray.Dataset: The loaded CDI dataset filtered to the specified end year
    """
    logger.info(f"Loading CDI file from {file_path}")
    
    ds = xr.open_dataset(file_path)
    
    # Convert time values to datetime for easier filtering
    times = pd.to_datetime(ds.time.values)
    logger.info(f"Original time range: {times[0]} to {times[-1]}")
    logger.info(f"Total time steps: {len(times)}")
    
    # Filter to include only data up to the specified end year
    time_mask = times.year <= end_year
    filtered_times = times[time_mask]
    
    logger.info(f"Filtering to data up to {end_year}")
    logger.info(f"Filtered time range: {filtered_times[0]} to {filtered_times[-1]}")
    logger.info(f"Filtered time steps: {len(filtered_times)}")
    
    # Create a new dataset with only the filtered time steps
    ds_filtered = ds.sel(time=ds.time[time_mask])
    
    return ds_filtered

def calculate_drought_percentages_for_month(cdi_data, drought_threshold=0.3):
    """
    Calculate drought percentages for a single month's CDI data.
    
    Args:
        cdi_data (numpy.ndarray): CDI values for a single month
        drought_threshold (float): Threshold for drought classification
        
    Returns:
        tuple: (drought_percentage_map, valid_mask, drought_count, total_valid)
    """
    # Create a mask for valid data (between 0 and 1, not NaN)
    valid_mask = ~np.isnan(cdi_data) & (cdi_data >= 0) & (cdi_data <= 1)
    
    # Calculate drought status for each cell
    drought_status = np.full(cdi_data.shape, np.nan, dtype=np.float32)
    drought_status[valid_mask] = (cdi_data[valid_mask] <= drought_threshold).astype(np.float32)
    
    # Count drought and total valid cells
    drought_count = np.sum(drought_status == 1)
    total_valid = np.sum(valid_mask)
    
    # Calculate percentage (multiply by 100 to get percentage)
    drought_percentage = (drought_count / total_valid * 100) if total_valid > 0 else 0
    
    # For visualization, we'll use the raw drought status (0 or 1) for each cell
    # but we'll also return the overall percentage for the title
    return drought_status, valid_mask, int(drought_count), int(total_valid), drought_percentage

def create_drought_colormap():
    """
    Create a custom colormap for drought visualization using warm colors.
    
    Returns:
        dict: Dictionary containing colormap information
    """
    # Define colors from white (no drought) to dark red (drought)
    # We'll use a gradient from white -> light yellow -> orange -> red -> dark red
    colors = [
        '#FFFFFF',  # White (0% - no drought)
        '#FFF8DC',  # Light cream (10%)
        '#FFE4B5',  # Light orange (20%)
        '#FFD700',  # Gold (30%)
        '#FFA500',  # Orange (40%)
        '#FF8C00',  # Dark orange (50%)
        '#FF4500',  # Red orange (60%)
        '#FF0000',  # Red (70%)
        '#DC143C',  # Crimson (80%)
        '#8B0000',  # Dark red (90%)
        '#800000'   # Maroon (100%)
    ]
    
    # Create the colormap
    drought_cmap = mcolors.LinearSegmentedColormap.from_list("drought", colors, N=256)
    
    # Define bounds for percentage-based coloring
    bounds = [0, 10, 20, 30, 40, 50, 60, 70, 80, 90, 100]
    norm = mcolors.BoundaryNorm(bounds, drought_cmap.N)
    
    return {
        'cmap': drought_cmap,
        'norm': norm,
        'bounds': bounds,
        'label': 'Drought Percentage (%)'
    }

def create_consistency_colormap():
    """
    Create a custom colormap for consistency visualization using warm colors.
    
    Returns:
        dict: Dictionary containing colormap information
    """
    # Define colors from light yellow (low consistency) to dark red (high consistency)
    colors = [
        '#FFFACD',  # Light yellow (0-20%)
        '#FFE4B5',  # Light orange (20-40%)
        '#FFA500',  # Orange (40-60%)
        '#FF4500',  # Red orange (60-80%)
        '#DC143C',  # Crimson (80-100%)
    ]
    
    # Create the colormap
    consistency_cmap = mcolors.ListedColormap(colors)
    
    # Define bounds for percentage-based coloring
    bounds = [0, 20, 40, 60, 80, 100]
    norm = mcolors.BoundaryNorm(bounds, consistency_cmap.N)
    
    return {
        'cmap': consistency_cmap,
        'norm': norm,
        'bounds': bounds,
        'label': 'Consistency Percentage (%)'
    }

def organize_data_by_year(ds, start_year=1998, end_year=2018):
    """
    Organize CDI data by year and month.
    
    Args:
        ds (xarray.Dataset): CDI dataset
        start_year (int): Start year
        end_year (int): End year
        
    Returns:
        dict: Dictionary organized as {year: {month: cdi_data}}
    """
    logger.info(f"Organizing data by year from {start_year} to {end_year}")
    
    times = pd.to_datetime(ds.time.values)
    data_by_year = {}
    
    for i, time in enumerate(times):
        year = time.year
        month = time.month
        
        if start_year <= year <= end_year:
            if year not in data_by_year:
                data_by_year[year] = {}
            
            data_by_year[year][month] = ds.cdi.values[:, :, i]
    
    logger.info(f"Organized data for {len(data_by_year)} years")
    return data_by_year

def save_drought_status_netcdf(data_by_year, latitude, longitude, output_file, drought_threshold=0.3):
    """
    Save drought status data to NetCDF file.
    
    Args:
        data_by_year (dict): Data organized by year and month
        latitude (numpy.ndarray): Latitude values
        longitude (numpy.ndarray): Longitude values
        output_file (str): Output NetCDF file path
        drought_threshold (float): Drought threshold
        
    Returns:
        str: Path to the saved NetCDF file
    """
    logger.info(f"Saving drought status data to NetCDF: {output_file}")
    
    # Create time coordinate
    time_stamps = []
    drought_data_list = []
    
    for year in sorted(data_by_year.keys()):
        for month in sorted(data_by_year[year].keys()):
            # Create timestamp for this month
            timestamp = pd.Timestamp(year=year, month=month, day=1)
            time_stamps.append(timestamp)
            
            # Calculate drought status
            cdi_data = data_by_year[year][month]
            drought_status, _, _, _, _ = calculate_drought_percentages_for_month(cdi_data, drought_threshold)
            drought_data_list.append(drought_status)
    
    # Stack the data
    drought_array = np.stack(drought_data_list, axis=2)
    
    # Create xarray Dataset
    ds_drought = xr.Dataset(
        {
            'drought_status': (['latitude', 'longitude', 'time'], drought_array),
        },
        coords={
            'latitude': latitude,
            'longitude': longitude,
            'time': time_stamps
        },
        attrs={
            'title': 'CDI Drought Status Analysis',
            'description': f'Drought status based on CDI threshold of {drought_threshold}',
            'drought_threshold': drought_threshold,
            'creation_date': datetime.now().isoformat(),
            'conventions': 'CF-1.6'
        }
    )
    
    # Add variable attributes
    ds_drought['drought_status'].attrs = {
        'long_name': 'Drought Status',
        'description': 'Binary drought status (1 = drought, 0 = no drought, NaN = invalid data)',
        'units': 'dimensionless',
        'drought_threshold': drought_threshold,
        'valid_range': [0, 1]
    }
    
    ds_drought['latitude'].attrs = {
        'long_name': 'Latitude',
        'units': 'degrees_north',
        'axis': 'Y'
    }
    
    ds_drought['longitude'].attrs = {
        'long_name': 'Longitude',
        'units': 'degrees_east',
        'axis': 'X'
    }
    
    ds_drought['time'].attrs = {
        'long_name': 'Time',
        'axis': 'T'
    }
    
    # Save to NetCDF
    os.makedirs(os.path.dirname(output_file), exist_ok=True)
    ds_drought.to_netcdf(output_file)
    
    logger.info(f"Drought status NetCDF saved: {output_file}")
    logger.info(f"Data shape: {drought_array.shape}")
    logger.info(f"Time range: {time_stamps[0]} to {time_stamps[-1]}")
    
    return output_file

#!/usr/bin/env python
# -*- coding: utf-8 -*-

"""
Phase 2: Map Generation Functions for Yearly Drought Maps
This phase includes functions for generating individual month maps and yearly combined maps.
"""

def generate_single_month_subplot(ax, longitude, latitude, data, month_name, year, 
                                colormap_info, drought_pct=None, map_type='drought'):
    """
    Generate a subplot for a single month.
    
    Args:
        ax: Matplotlib axis object
        longitude (numpy.ndarray): Longitude values
        latitude (numpy.ndarray): Latitude values
        data (numpy.ndarray): Data to plot
        month_name (str): Name of the month
        year (int): Year
        colormap_info (dict): Colormap information
        drought_pct (float): Overall drought percentage for the month
        map_type (str): Type of map ('drought' or 'consistency')
    """
    # Set the extent to focus on Australia
    ax.set_extent([110, 155, -45, -10], crs=ccrs.PlateCarree())
    
    # Add coastlines
    ax.coastlines(resolution='50m', color='black', linewidth=0.3)
    ax.add_feature(cfeature.BORDERS, linewidth=0.2, linestyle=':')
    
    # Plot the data
    im = ax.pcolormesh(longitude, latitude, data, 
                      cmap=colormap_info['cmap'], 
                      norm=colormap_info['norm'], 
                      transform=ccrs.PlateCarree())
    
    # Add title with percentage if available
    if drought_pct is not None:
        title = f"{month_name}\n{drought_pct:.1f}% Drought"
    else:
        title = f"{month_name}"
    
    ax.set_title(title, fontsize=10, pad=5)
    
    # Add very light gridlines
    gl = ax.gridlines(crs=ccrs.PlateCarree(), draw_labels=False,
                     linewidth=0.2, color='gray', alpha=0.5, linestyle=':')
    
    return im

def generate_yearly_drought_map(year, year_data, latitude, longitude, 
                               output_dir, drought_threshold=0.3):
    """
    Generate a yearly map showing all months with drought percentages.
    
    Args:
        year (int): Year to process
        year_data (dict): Dictionary with month number as key and CDI data as value
        latitude (numpy.ndarray): Latitude values
        longitude (numpy.ndarray): Longitude values
        output_dir (str): Output directory
        drought_threshold (float): Drought threshold
        
    Returns:
        str: Path to the generated map
    """
    logger.info(f"Generating yearly drought map for {year}")
    
    # Create output directory if it doesn't exist
    os.makedirs(output_dir, exist_ok=True)
    
    # Sort months
    available_months = sorted(year_data.keys())
    num_months = len(available_months)
    
    if num_months == 0:
        logger.warning(f"No data available for year {year}")
        return None
    
    # Determine subplot layout (prefer 4 columns for better aspect ratio)
    if num_months <= 4:
        rows, cols = 1, num_months
    elif num_months <= 8:
        rows, cols = 2, 4
    else:
        rows, cols = 3, 4
    
    # Create the figure with subplots
    fig = plt.figure(figsize=(cols * 4, rows * 3.5))
    fig.suptitle(f'Drought Analysis for {year}', fontsize=16, y=0.95)
    
    # Create colormap for drought visualization
    colormap_info = create_drought_colormap()
    
    # Store overall statistics
    year_stats = []
    
    # Process each month
    for i, month_num in enumerate(available_months):
        month_name = calendar.month_name[month_num]
        cdi_data = year_data[month_num]
        
        # Calculate drought information
        drought_status, valid_mask, drought_count, total_valid, drought_pct = \
            calculate_drought_percentages_for_month(cdi_data, drought_threshold)
        
        year_stats.append({
            'month': month_name,
            'month_num': month_num,
            'drought_count': drought_count,
            'total_valid': total_valid,
            'drought_pct': drought_pct
        })
        
        # Create subplot
        ax = fig.add_subplot(rows, cols, i+1, projection=ccrs.PlateCarree())
        
        # For visualization, we want to show drought percentage by cell
        # Convert drought status to percentage representation for better visualization
        drought_viz_data = np.full(drought_status.shape, np.nan, dtype=np.float32)
        drought_viz_data[valid_mask] = drought_status[valid_mask] * 100  # 0 or 100 for binary
        
        # Generate the subplot
        im = generate_single_month_subplot(
            ax, longitude, latitude, drought_viz_data, 
            month_name, year, colormap_info, drought_pct, 'drought'
        )
    
    # Add a colorbar at the bottom
    cbar_ax = fig.add_axes([0.15, 0.05, 0.7, 0.02])
    cbar = fig.colorbar(im, cax=cbar_ax, orientation='horizontal')
    cbar.set_label('Drought Status (%)', fontsize=12)
    cbar.set_ticks([0, 25, 50, 75, 100])
    cbar.set_ticklabels(['No Drought', '25%', '50%', '75%', 'Drought'])
    
    # Add summary statistics text
    total_drought_cells = sum(stat['drought_count'] for stat in year_stats)
    total_valid_cells = sum(stat['total_valid'] for stat in year_stats)
    overall_drought_pct = (total_drought_cells / total_valid_cells * 100) if total_valid_cells > 0 else 0
    
    stats_text = f"Year {year} Summary: {overall_drought_pct:.1f}% overall drought"
    fig.text(0.5, 0.02, stats_text, ha='center', fontsize=12, 
             bbox=dict(boxstyle="round,pad=0.3", facecolor="white", alpha=0.8))
    
    # Save the figure
    output_file = os.path.join(output_dir, f"drought_analysis_{year}.png")
    plt.savefig(output_file, dpi=300, bbox_inches='tight', facecolor='white')
    plt.close(fig)
    
    logger.info(f"Saved yearly drought map: {output_file}")
    return output_file

def generate_yearly_consistency_map(year, consistency_data, latitude, longitude, 
                                  output_dir):
    """
    Generate a yearly map showing consistency percentages for all months.
    
    Args:
        year (int): Year to process
        consistency_data (dict): Dictionary with month number as key and consistency data as value
        latitude (numpy.ndarray): Latitude values
        longitude (numpy.ndarray): Longitude values
        output_dir (str): Output directory
        
    Returns:
        str: Path to the generated map
    """
    logger.info(f"Generating yearly consistency map for {year}")
    
    # Create output directory if it doesn't exist
    os.makedirs(output_dir, exist_ok=True)
    
    # Sort months
    available_months = sorted(consistency_data.keys())
    num_months = len(available_months)
    
    if num_months == 0:
        logger.warning(f"No consistency data available for year {year}")
        return None
    
    # Determine subplot layout
    if num_months <= 4:
        rows, cols = 1, num_months
    elif num_months <= 8:
        rows, cols = 2, 4
    else:
        rows, cols = 3, 4
    
    # Create the figure with subplots
    fig = plt.figure(figsize=(cols * 4, rows * 3.5))
    fig.suptitle(f'Drought Consistency Analysis for {year}', fontsize=16, y=0.95)
    
    # Create colormap for consistency visualization
    colormap_info = create_consistency_colormap()
    
    # Process each month
    for i, month_num in enumerate(available_months):
        month_name = calendar.month_name[month_num]
        consistency_pct = consistency_data[month_num]
        
        # Create subplot
        ax = fig.add_subplot(rows, cols, i+1, projection=ccrs.PlateCarree())
        
        # Calculate mean consistency for title
        valid_consistency = consistency_pct[~np.isnan(consistency_pct)]
        mean_consistency = np.mean(valid_consistency) if len(valid_consistency) > 0 else 0
        
        # Generate the subplot
        im = generate_single_month_subplot(
            ax, longitude, latitude, consistency_pct, 
            month_name, year, colormap_info, mean_consistency, 'consistency'
        )
    
    # Add a colorbar at the bottom
    cbar_ax = fig.add_axes([0.15, 0.05, 0.7, 0.02])
    cbar = fig.colorbar(im, cax=cbar_ax, orientation='horizontal')
    cbar.set_label('Consistency Percentage (%)', fontsize=12)
    cbar.set_ticks([10, 30, 50, 70, 90])
    
    # Save the figure
    output_file = os.path.join(output_dir, f"consistency_analysis_{year}.png")
    plt.savefig(output_file, dpi=300, bbox_inches='tight', facecolor='white')
    plt.close(fig)
    
    logger.info(f"Saved yearly consistency map: {output_file}")
    return output_file

#!/usr/bin/env python
# -*- coding: utf-8 -*-

"""
Phase 3: Main Processing and Consistency Calculations with NetCDF Output
This phase includes the main processing functions and consistency calculations.
"""

def calculate_monthly_consistency(data_by_year, drought_threshold=0.3):
    """
    Calculate consistency data for each month across all years.
    
    Args:
        data_by_year (dict): Data organized by year and month
        drought_threshold (float): Drought threshold
        
    Returns:
        dict: Consistency data organized by year and month
    """
    logger.info("Calculating monthly consistency across all years")
    
    # First, reorganize data by month across years
    data_by_month = {}
    for year, year_data in data_by_year.items():
        for month, month_data in year_data.items():
            if month not in data_by_month:
                data_by_month[month] = {}
            data_by_month[month][year] = month_data
    
    # Calculate consistency for each month
    consistency_by_year = {}
    
    for month_num in range(1, 13):  # Process all 12 months
        if month_num not in data_by_month:
            continue
            
        month_name = calendar.month_name[month_num]
        logger.info(f"Processing consistency for {month_name}")
        
        # Get next month number (handle December -> January)
        next_month_num = (month_num % 12) + 1
        
        if next_month_num not in data_by_month:
            continue
        
        # Get data for current and next month
        current_month_data = data_by_month[month_num]
        next_month_data = data_by_month[next_month_num]
        
        # Find common years
        current_years = set(current_month_data.keys())
        next_years = set(next_month_data.keys())
        
        # Handle year transition for December-January
        if month_num == 12:  # December
            # For December, next month is January of the following year
            common_pairs = []
            for year in current_years:
                if (year + 1) in next_years:
                    common_pairs.append((year, year + 1))
        else:
            # For other months, same year
            common_years = current_years & next_years
            common_pairs = [(year, year) for year in common_years]
        
        if not common_pairs:
            continue
        
        # Get shape from first available data
        first_year = list(current_month_data.keys())[0]
        shape = current_month_data[first_year].shape
        
        # Initialize consistency calculation arrays
        consistency_sum = np.zeros(shape, dtype=np.float32)
        valid_count = np.zeros(shape, dtype=np.int32)
        
        # Process each year pair
        for curr_year, next_year in common_pairs:
            curr_data = current_month_data[curr_year]
            next_data = next_month_data[next_year]
            
            # Create valid masks
            curr_valid = ~np.isnan(curr_data) & (curr_data >= 0) & (curr_data <= 1)
            next_valid = ~np.isnan(next_data) & (next_data >= 0) & (next_data <= 1)
            both_valid = curr_valid & next_valid
            
            if np.any(both_valid):
                # Calculate drought status
                curr_drought = (curr_data <= drought_threshold).astype(np.float32)
                next_drought = (next_data <= drought_threshold).astype(np.float32)
                
                # Calculate consistency (1 if same status, 0 if different)
                is_consistent = (curr_drought == next_drought).astype(np.float32)
                
                # Update sums
                consistency_sum[both_valid] += is_consistent[both_valid]
                valid_count[both_valid] += 1
        
        # Calculate consistency percentage
        consistency_percentage = np.full(shape, np.nan, dtype=np.float32)
        nonzero_mask = (valid_count > 0)
        consistency_percentage[nonzero_mask] = (consistency_sum[nonzero_mask] / valid_count[nonzero_mask]) * 100.0
        
        # Store consistency data for each year that has this month
        for year in current_month_data.keys():
            if year not in consistency_by_year:
                consistency_by_year[year] = {}
            consistency_by_year[year][month_num] = consistency_percentage
    
    logger.info(f"Calculated consistency for {len(consistency_by_year)} years")
    return consistency_by_year

def save_consistency_netcdf(consistency_by_year, latitude, longitude, output_file):
    """
    Save consistency data to NetCDF file.
    
    Args:
        consistency_by_year (dict): Consistency data organized by year and month
        latitude (numpy.ndarray): Latitude values
        longitude (numpy.ndarray): Longitude values
        output_file (str): Output NetCDF file path
        
    Returns:
        str: Path to the saved NetCDF file
    """
    logger.info(f"Saving consistency data to NetCDF: {output_file}")
    
    # Create time coordinate and collect data
    time_stamps = []
    consistency_data_list = []
    
    for year in sorted(consistency_by_year.keys()):
        for month in sorted(consistency_by_year[year].keys()):
            # Create timestamp for this month
            timestamp = pd.Timestamp(year=year, month=month, day=1)
            time_stamps.append(timestamp)
            
            # Get consistency data
            consistency_data = consistency_by_year[year][month]
            consistency_data_list.append(consistency_data)
    
    # Stack the data
    consistency_array = np.stack(consistency_data_list, axis=2)
    
    # Create xarray Dataset
    ds_consistency = xr.Dataset(
        {
            'consistency_percentage': (['latitude', 'longitude', 'time'], consistency_array),
        },
        coords={
            'latitude': latitude,
            'longitude': longitude,
            'time': time_stamps
        },
        attrs={
            'title': 'CDI Drought Consistency Analysis',
            'description': 'Drought consistency percentages between consecutive months based on historical patterns',
            'creation_date': datetime.now().isoformat(),
            'conventions': 'CF-1.6',
            'analysis_period': '1998-2018'
        }
    )
    
    # Add variable attributes
    ds_consistency['consistency_percentage'].attrs = {
        'long_name': 'Drought Consistency Percentage',
        'description': 'Percentage of years where drought status remained consistent between current and next month',
        'units': 'percent',
        'valid_range': [0, 100],
        'methodology': 'Based on consecutive month comparison across 1998-2018 period'
    }
    
    ds_consistency['latitude'].attrs = {
        'long_name': 'Latitude',
        'units': 'degrees_north',
        'axis': 'Y'
    }
    
    ds_consistency['longitude'].attrs = {
        'long_name': 'Longitude',
        'units': 'degrees_east',
        'axis': 'X'
    }
    
    ds_consistency['time'].attrs = {
        'long_name': 'Time',
        'axis': 'T'
    }
    
    # Save to NetCDF
    os.makedirs(os.path.dirname(output_file), exist_ok=True)
    ds_consistency.to_netcdf(output_file)
    
    logger.info(f"Consistency NetCDF saved: {output_file}")
    logger.info(f"Data shape: {consistency_array.shape}")
    logger.info(f"Time range: {time_stamps[0]} to {time_stamps[-1]}")
    
    return output_file

def save_combined_analysis_netcdf(data_by_year, consistency_by_year, latitude, longitude, 
                                 output_file, drought_threshold=0.3):
    """
    Save combined drought and consistency analysis to a single NetCDF file.
    
    Args:
        data_by_year (dict): CDI data organized by year and month
        consistency_by_year (dict): Consistency data organized by year and month
        latitude (numpy.ndarray): Latitude values
        longitude (numpy.ndarray): Longitude values
        output_file (str): Output NetCDF file path
        drought_threshold (float): Drought threshold
        
    Returns:
        str: Path to the saved NetCDF file
    """
    logger.info(f"Saving combined analysis data to NetCDF: {output_file}")
    
    # Create time coordinate and collect data
    time_stamps = []
    drought_data_list = []
    consistency_data_list = []
    original_cdi_list = []
    
    for year in sorted(data_by_year.keys()):
        for month in sorted(data_by_year[year].keys()):
            # Create timestamp for this month
            timestamp = pd.Timestamp(year=year, month=month, day=1)
            time_stamps.append(timestamp)
            
            # Get original CDI data
            cdi_data = data_by_year[year][month]
            original_cdi_list.append(cdi_data)
            
            # Calculate drought status
            drought_status, _, _, _, _ = calculate_drought_percentages_for_month(cdi_data, drought_threshold)
            drought_data_list.append(drought_status)
            
            # Get consistency data if available
            if year in consistency_by_year and month in consistency_by_year[year]:
                consistency_data = consistency_by_year[year][month]
            else:
                # Create NaN array with same shape as CDI data
                consistency_data = np.full(cdi_data.shape, np.nan, dtype=np.float32)
            consistency_data_list.append(consistency_data)
    
    # Stack the data
    original_cdi_array = np.stack(original_cdi_list, axis=2)
    drought_array = np.stack(drought_data_list, axis=2)
    consistency_array = np.stack(consistency_data_list, axis=2)
    
    # Create xarray Dataset
    ds_combined = xr.Dataset(
        {
            'cdi': (['latitude', 'longitude', 'time'], original_cdi_array),
            'drought_status': (['latitude', 'longitude', 'time'], drought_array),
            'consistency_percentage': (['latitude', 'longitude', 'time'], consistency_array),
        },
        coords={
            'latitude': latitude,
            'longitude': longitude,
            'time': time_stamps
        },
        attrs={
            'title': 'Combined CDI Drought Analysis',
            'description': 'Combined analysis including original CDI, drought status, and consistency percentages',
            'drought_threshold': drought_threshold,
            'creation_date': datetime.now().isoformat(),
            'conventions': 'CF-1.6',
            'analysis_period': '1998-2018',
            'methodology': 'CDI-based drought analysis with consecutive month consistency calculation'
        }
    )
    
    # Add variable attributes
    ds_combined['cdi'].attrs = {
        'long_name': 'Combined Drought Index',
        'description': 'Original CDI values',
        'units': 'dimensionless',
        'valid_range': [0, 1]
    }
    
    ds_combined['drought_status'].attrs = {
        'long_name': 'Drought Status',
        'description': f'Binary drought status (1 = drought, 0 = no drought, NaN = invalid data) based on CDI <= {drought_threshold}',
        'units': 'dimensionless',
        'drought_threshold': drought_threshold,
        'valid_range': [0, 1]
    }
    
    ds_combined['consistency_percentage'].attrs = {
        'long_name': 'Drought Consistency Percentage',
        'description': 'Percentage of years where drought status remained consistent between current and next month',
        'units': 'percent',
        'valid_range': [0, 100],
        'methodology': 'Based on consecutive month comparison across 1998-2018 period'
    }
    
    ds_combined['latitude'].attrs = {
        'long_name': 'Latitude',
        'units': 'degrees_north',
        'axis': 'Y'
    }
    
    ds_combined['longitude'].attrs = {
        'long_name': 'Longitude',
        'units': 'degrees_east',
        'axis': 'X'
    }
    
    ds_combined['time'].attrs = {
        'long_name': 'Time',
        'axis': 'T'
    }
    
    # Save to NetCDF
    os.makedirs(os.path.dirname(output_file), exist_ok=True)
    ds_combined.to_netcdf(output_file)
    
    logger.info(f"Combined analysis NetCDF saved: {output_file}")
    logger.info(f"Data shape: {original_cdi_array.shape}")
    logger.info(f"Time range: {time_stamps[0]} to {time_stamps[-1]}")
    logger.info(f"Variables saved: cdi, drought_status, consistency_percentage")
    
    return output_file

def process_all_years(data_by_year, consistency_by_year, latitude, longitude, 
                     output_base_dir, drought_threshold=0.3):
    """
    Process all years to generate drought and consistency maps, plus NetCDF files.
    
    Args:
        data_by_year (dict): CDI data organized by year and month
        consistency_by_year (dict): Consistency data organized by year and month
        latitude (numpy.ndarray): Latitude values
        longitude (numpy.ndarray): Longitude values
        output_base_dir (str): Base output directory
        drought_threshold (float): Drought threshold
        
    Returns:
        dict: Dictionary with processing results
    """
    logger.info("Processing all years to generate maps and NetCDF files")
    
    # Create output directories
    drought_dir = os.path.join(output_base_dir, "drought_maps")
    consistency_dir = os.path.join(output_base_dir, "consistency_maps")
    netcdf_dir = os.path.join(output_base_dir, "netcdf_outputs")
    os.makedirs(drought_dir, exist_ok=True)
    os.makedirs(consistency_dir, exist_ok=True)
    os.makedirs(netcdf_dir, exist_ok=True)
    
    results = {
        'drought_maps': [],
        'consistency_maps': [],
        'netcdf_files': [],
        'years_processed': [],
        'total_maps': 0
    }
    
    # Process each year for maps
    years = sorted(data_by_year.keys())
    logger.info(f"Processing {len(years)} years for maps: {years}")
    
    for year in years:
        logger.info(f"Processing year {year}")
        
        # Generate drought map for this year
        drought_map_path = generate_yearly_drought_map(
            year, data_by_year[year], latitude, longitude, 
            drought_dir, drought_threshold
        )
        
        if drought_map_path:
            results['drought_maps'].append(drought_map_path)
            results['total_maps'] += 1
        
        # Generate consistency map for this year if data available
        if year in consistency_by_year:
            consistency_map_path = generate_yearly_consistency_map(
                year, consistency_by_year[year], latitude, longitude, 
                consistency_dir
            )
            
            if consistency_map_path:
                results['consistency_maps'].append(consistency_map_path)
                results['total_maps'] += 1
        
        results['years_processed'].append(year)
    
    # Generate NetCDF files
    logger.info("Generating NetCDF files...")
    
    # Save drought status NetCDF
    drought_netcdf = os.path.join(netcdf_dir, "drought_status_analysis.nc")
    drought_nc_path = save_drought_status_netcdf(
        data_by_year, latitude, longitude, drought_netcdf, drought_threshold
    )
    results['netcdf_files'].append(drought_nc_path)
    
    # Save consistency NetCDF
    consistency_netcdf = os.path.join(netcdf_dir, "consistency_analysis.nc")
    consistency_nc_path = save_consistency_netcdf(
        consistency_by_year, latitude, longitude, consistency_netcdf
    )
    results['netcdf_files'].append(consistency_nc_path)
    
    # Save combined analysis NetCDF
    combined_netcdf = os.path.join(netcdf_dir, "combined_drought_analysis.nc")
    combined_nc_path = save_combined_analysis_netcdf(
        data_by_year, consistency_by_year, latitude, longitude, 
        combined_netcdf, drought_threshold
    )
    results['netcdf_files'].append(combined_nc_path)
    
    logger.info(f"Generated {results['total_maps']} maps and {len(results['netcdf_files'])} NetCDF files for {len(results['years_processed'])} years")
    return results

#!/usr/bin/env python
# -*- coding: utf-8 -*-

"""
Phase 4: Main Execution Function with NetCDF Output
This phase includes the main execution function that generates both maps and NetCDF files.
"""

def create_summary_report(results, output_file="yearly_maps_summary.txt"):
    """
    Create a summary report of the generated maps and NetCDF files.
    
    Args:
        results (dict): Processing results
        output_file (str): Output file path
        
    Returns:
        str: Path to the summary file
    """
    logger.info(f"Creating summary report: {output_file}")
    
    with open(output_file, 'w') as f:
        f.write("=" * 80 + "\n")
        f.write("YEARLY DROUGHT MAPS AND NETCDF GENERATION SUMMARY\n")
        f.write("=" * 80 + "\n\n")
        
        f.write("PROCESSING DETAILS:\n")
        f.write("-" * 50 + "\n")
        f.write(f"Date processed: {datetime.now().strftime('%Y-%m-%d %H:%M:%S')}\n")
        f.write(f"Total maps generated: {results['total_maps']}\n")
        f.write(f"Total NetCDF files generated: {len(results['netcdf_files'])}\n")
        f.write(f"Years processed: {len(results['years_processed'])}\n")
        f.write(f"Drought maps: {len(results['drought_maps'])}\n")
        f.write(f"Consistency maps: {len(results['consistency_maps'])}\n\n")
        
        f.write("YEARS PROCESSED:\n")
        f.write("-" * 50 + "\n")
        for year in sorted(results['years_processed']):
            f.write(f"  {year}\n")
        f.write("\n")
        
        f.write("NETCDF FILES GENERATED:\n")
        f.write("-" * 50 + "\n")
        for netcdf_path in results['netcdf_files']:
            f.write(f"  {os.path.basename(netcdf_path)}\n")
        f.write("\n")
        
        f.write("DROUGHT MAPS GENERATED:\n")
        f.write("-" * 50 + "\n")
        for map_path in results['drought_maps']:
            f.write(f"  {os.path.basename(map_path)}\n")
        f.write("\n")
        
        f.write("CONSISTENCY MAPS GENERATED:\n")
        f.write("-" * 50 + "\n")
        for map_path in results['consistency_maps']:
            f.write(f"  {os.path.basename(map_path)}\n")
        f.write("\n")
        
        f.write("NETCDF FILE DESCRIPTIONS:\n")
        f.write("-" * 50 + "\n")
        f.write("1. drought_status_analysis.nc:\n")
        f.write("   - Contains binary drought status data (0/1) for each time step\n")
        f.write("   - Variables: drought_status\n")
        f.write("   - Based on CDI threshold of 0.3\n")
        f.write("   - Dimensions: [latitude, longitude, time]\n\n")
        
        f.write("2. consistency_analysis.nc:\n")
        f.write("   - Contains consistency percentages between consecutive months\n")
        f.write("   - Variables: consistency_percentage\n")
        f.write("   - Based on historical patterns (1998-2018)\n")
        f.write("   - Values range from 0-100%\n")
        f.write("   - Dimensions: [latitude, longitude, time]\n\n")
        
        f.write("3. combined_drought_analysis.nc:\n")
        f.write("   - Contains all variables in one file\n")
        f.write("   - Variables: cdi, drought_status, consistency_percentage\n")
        f.write("   - Most comprehensive dataset for analysis\n")
        f.write("   - Dimensions: [latitude, longitude, time]\n\n")
        
        f.write("MAP DESCRIPTIONS:\n")
        f.write("-" * 50 + "\n")
        f.write("1. Drought Maps:\n")
        f.write("   - Show actual drought conditions for each month of each year\n")
        f.write("   - Red colors indicate drought conditions (CDI ≤ 0.3)\n")
        f.write("   - White/light colors indicate no drought conditions\n")
        f.write("   - Each subplot shows one month with drought percentage\n")
        f.write("   - Overall year summary included at bottom\n\n")
        
        f.write("2. Consistency Maps:\n")
        f.write("   - Show historical consistency between consecutive months\n")
        f.write("   - Warm colors (yellow to red) indicate higher consistency\n")
        f.write("   - Based on historical patterns from 1998-2018\n")
        f.write("   - Higher values mean drought status typically remains consistent\n\n")
        
        f.write("USAGE NOTES:\n")
        f.write("-" * 50 + "\n")
        f.write("1. NetCDF files can be opened with xarray, netCDF4, or similar tools\n")
        f.write("2. Drought maps show actual conditions for specific years\n")
        f.write("3. Consistency maps show long-term patterns and predictability\n")
        f.write("4. Use combined_drought_analysis.nc for comprehensive analysis\n")
        f.write("5. NetCDF files preserve all metadata and coordinate information\n")
        f.write("6. Files are CF-1.6 compliant for broad compatibility\n\n")
        
        f.write("EXAMPLE USAGE:\n")
        f.write("-" * 50 + "\n")
        f.write("# Load with xarray:\n")
        f.write("import xarray as xr\n")
        f.write("ds = xr.open_dataset('combined_drought_analysis.nc')\n")
        f.write("print(ds.info())\n\n")
        f.write("# Access drought status for a specific time:\n")
        f.write("drought_jan_2000 = ds.drought_status.sel(time='2000-01-01')\n\n")
        f.write("# Calculate mean consistency:\n")
        f.write("mean_consistency = ds.consistency_percentage.mean(dim='time')\n\n")
        
        f.write("=" * 80 + "\n")
        f.write("End of Summary\n")
        f.write("=" * 80 + "\n")
    
    logger.info(f"Summary report created: {output_file}")
    return output_file

def main():
    """
    Main function to generate yearly drought maps and NetCDF files.
    """
    start_time = datetime.now()
    logger.info("=" * 80)
    logger.info("YEARLY DROUGHT MAPS AND NETCDF GENERATION")
    logger.info("=" * 80)
    
    # Configuration
    config = {
        'cdi_file': "/Volumes/data/nacp/results/netcdf/cdi_1.nc",  # Path to your CDI file
        'output_base_dir': "yearly_drought_analysis",  # Base output directory
        'start_year': 1998,  # Start year for analysis
        'end_year': 2018,   # End year for analysis
        'drought_threshold': 0.3,  # Threshold for drought classification
        'summary_file': "drought_analysis_summary.txt"  # Summary report file
    }
    
    try:
        logger.info("Starting yearly drought analysis with NetCDF output...")
        logger.info(f"Configuration: {config}")
        
        # Step 1: Load CDI data
        logger.info("Step 1: Loading CDI data...")
        ds = load_cdi_data(config['cdi_file'], config['end_year'])
        
        # Extract coordinates
        latitude = ds.latitude.values
        longitude = ds.longitude.values
        
        logger.info(f"Data loaded - Shape: {ds.cdi.shape}")
        logger.info(f"Latitude range: {latitude.min():.2f} to {latitude.max():.2f}")
        logger.info(f"Longitude range: {longitude.min():.2f} to {longitude.max():.2f}")
        
        # Step 2: Organize data by year and month
        logger.info("Step 2: Organizing data by year and month...")
        data_by_year = organize_data_by_year(ds, config['start_year'], config['end_year'])
        
        # Log data availability
        logger.info("Data availability by year:")
        for year in sorted(data_by_year.keys()):
            months = sorted(data_by_year[year].keys())
            month_names = [calendar.month_abbr[m] for m in months]
            logger.info(f"  {year}: {len(months)} months - {', '.join(month_names)}")
        
        # Step 3: Calculate monthly consistency
        logger.info("Step 3: Calculating monthly consistency...")
        consistency_by_year = calculate_monthly_consistency(
            data_by_year, 
            config['drought_threshold']
        )
        
        # Step 4: Generate maps and NetCDF files for all years
        logger.info("Step 4: Generating maps and NetCDF files for all years...")
        results = process_all_years(
            data_by_year, 
            consistency_by_year, 
            latitude, 
            longitude,
            config['output_base_dir'],
            config['drought_threshold']
        )
        
        # Step 5: Create summary report
        logger.info("Step 5: Creating summary report...")
        summary_file = create_summary_report(results, config['summary_file'])
        
        # Log completion
        end_time = datetime.now()
        duration = (end_time - start_time).total_seconds()
        
        logger.info("=" * 80)
        logger.info(f"PROCESSING COMPLETED IN {duration:.1f} SECONDS")
        logger.info("=" * 80)
        logger.info(f"Total maps generated: {results['total_maps']}")
        logger.info(f"Total NetCDF files generated: {len(results['netcdf_files'])}")
        logger.info(f"Years processed: {len(results['years_processed'])}")
        logger.info(f"Output directory: {config['output_base_dir']}")
        logger.info(f"Summary report: {summary_file}")
        logger.info("=" * 80)
        
        # Print file locations for easy access
        print("\n" + "=" * 80)
        print("FILES GENERATED:")
        print("=" * 80)
        print(f"Drought maps directory: {os.path.join(config['output_base_dir'], 'drought_maps')}")
        print(f"Consistency maps directory: {os.path.join(config['output_base_dir'], 'consistency_maps')}")
        print(f"NetCDF files directory: {os.path.join(config['output_base_dir'], 'netcdf_outputs')}")
        print(f"Summary report: {summary_file}")
        print("=" * 80)
        print("\nNetCDF FILES CREATED:")
        for nc_file in results['netcdf_files']:
            print(f"  - {nc_file}")
        print("=" * 80)
        
        return results
        
    except Exception as e:
        logger.error(f"Error occurred: {str(e)}")
        import traceback
        logger.error(traceback.format_exc())
        raise

def verify_netcdf_files(results):
    """
    Verify that the generated NetCDF files can be opened and contain expected data.
    
    Args:
        results (dict): Processing results containing NetCDF file paths
    """
    logger.info("Verifying generated NetCDF files...")
    
    for nc_file in results['netcdf_files']:
        try:
            logger.info(f"Verifying: {os.path.basename(nc_file)}")
            ds = xr.open_dataset(nc_file)
            
            logger.info(f"  Shape: {ds.dims}")
            logger.info(f"  Variables: {list(ds.data_vars.keys())}")
            logger.info(f"  Time range: {ds.time.values[0]} to {ds.time.values[-1]}")
            logger.info(f"  Coordinates: {list(ds.coords.keys())}")
            
            # Check for NaN values
            for var_name in ds.data_vars:
                var_data = ds[var_name].values
                total_values = var_data.size
                nan_count = np.isnan(var_data).sum()
                valid_count = total_values - nan_count
                logger.info(f"  {var_name}: {valid_count}/{total_values} valid values ({100*valid_count/total_values:.1f}%)")
            
            ds.close()
            logger.info(f"  ✓ File verified successfully")
            
        except Exception as e:
            logger.error(f"  ✗ Error verifying {nc_file}: {str(e)}")
    
    logger.info("NetCDF verification completed")

if __name__ == "__main__":
    # Run the main analysis
    results = main()
    
    # Verify the generated NetCDF files
    verify_netcdf_files(results)
    
    print("\n" + "=" * 80)
    print("ANALYSIS COMPLETE!")
    print("Check the log file 'yearly_drought_maps.log' for detailed processing information.")
    print("=" * 80)

2025-05-22 11:18:40 [INFO] ================================================================================
2025-05-22 11:18:40 [INFO] YEARLY DROUGHT MAPS AND NETCDF GENERATION
2025-05-22 11:18:40 [INFO] ================================================================================
2025-05-22 11:18:40 [INFO] Starting yearly drought analysis with NetCDF output...
2025-05-22 11:18:40 [INFO] Configuration: {'cdi_file': '/Volumes/data/nacp/results/netcdf/cdi_1.nc', 'output_base_dir': 'yearly_drought_analysis', 'start_year': 1998, 'end_year': 2018, 'drought_threshold': 0.3, 'summary_file': 'drought_analysis_summary.txt'}
2025-05-22 11:18:40 [INFO] Step 1: Loading CDI data...
2025-05-22 11:18:40 [INFO] Loading CDI file from /Volumes/data/nacp/results/netcdf/cdi_1.nc
2025-05-22 11:18:41 [INFO] Original time range: 1998-04-01 00:00:00 to 2025-04-01 00:00:00
2025-05-22 11:18:41 [INFO] Total time steps: 319
2025-05-22 11:18:41 [INFO] Filtering to data up to 2018
2025-05-22 11:18:41 [INFO] Filt